# MFA Alignment + Prosodic Analysis — PRESEEA-Valencia

**Requisitos previos:**
- `whisper_cache_180.json` en Drive (del notebook de transcripción)
- `colab_segments/` con los `_FULL.wav` en Drive

**Flujo:**
1. Ejecuta **Celda 1a** → el runtime se reinicia solo
2. Ejecuta **Celda 1b** en adelante (no vuelvas a ejecutar 1a)

**Runtime recomendado:** CPU (MFA no usa GPU)

In [1]:
# ── CELDA 1a: instalar condacolab ─────────────────────────────────────────────
# ⚠️  El runtime se reinicia automáticamente al terminar.
#     Después del reinicio, continúa desde la CELDA 1b.

!pip install condacolab -q
import condacolab
condacolab.install()  # ← reinicia el runtime

✨🍰✨ Everything looks OK!


In [2]:
# ── CELDA 1b: instalar MFA y dependencias (ejecutar TRAS el reinicio) ─────────
# MFA vía conda-forge: incluye Kaldi y todas las dependencias C++
# Sin whisperx ni pyannote → sin conflictos de versiones

!conda install -c conda-forge montreal-forced-aligner -y -q
!mfa model download dictionary spanish_mfa
!mfa model download acoustic  spanish_mfa
!pip install praat-parselmouth praatio soundfile librosa -q

print('✅ Instalación completada')
!mfa version

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - montreal-forced-aligner


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    adwaita-icon-theme-49.0    |           unix_0         617 KB  conda-forge
    aom-3.9.1                  |       hac33072_0         2.6 MB  conda-forge
    at-spi2-atk-2.38.0         |       h0630a04_3         332 KB  conda-forge
    at-spi2-core-2.40.3        |       h0630a04_0         643 KB  conda-forge
    atk-1.0-2.38.0             |       h04ea711_2         348 KB  conda-forge
    audioread-3.0.1            |  py311h38be061_3          44 KB  conda-forge
    baumwelch-0.3.11           |       hb700be7_1         402 KB  conda-forge
    brotli-1.1.0               |       hb9d3cd8_2          19 KB  conda-forge
    brotli-bin-1.1.0           |  

In [3]:
# ── CELDA 2: instalar paquetes Python + montar Drive ─────────────────────────
# subprocess.check_call instala en el proceso Python activo → no necesita reinicio

import subprocess, sys

pkgs = ['praat-parselmouth', 'praatio', 'soundfile', 'librosa']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('✅ Paquetes Python instalados')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json, csv, gc, re, shutil
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import soundfile as sf
import parselmouth
from praatio import textgrid as tg_module
from praatio.data_classes.interval_tier import IntervalTier

print('✅ Imports OK')

# ── ▼ AJUSTA ESTA RUTA ────────────────────────────────────────────────────────
DRIVE_BASE    = Path('/content/drive/MyDrive/EFE2025')
# ── ▲ ─────────────────────────────────────────────────────────────────────────

WHISPER_CACHE = DRIVE_BASE / 'prueba_Valentin.json'
SEGMENTS_DIR  = DRIVE_BASE / 'prueba_Valentin'
CSV_DIR       = DRIVE_BASE / 'por_hablante'
TG_DIR        = DRIVE_BASE / 'textgrids_Valentin'

MFA_BIN         = 'mfa'
MFA_DICTIONARY  = 'spanish_mfa'
MFA_ACOUSTIC    = 'spanish_mfa'
MFA_JOBS        = 2
START_PCT       = 0.0
END_PCT         = 100.0
PAUSE_THRESHOLD = 0.23
CONFIDENCE_THR  = 0.70
PITCH_FLOOR     = 75
PITCH_CEILING   = 500
P_LOW, P_HIGH   = 10, 90
MIN_VOICED      = 5
VOWEL_RE        = re.compile(r'^[aeiouAEIOU]')

LOCAL_CORPUS  = Path('/tmp/mfa_corpus')
LOCAL_ALIGNED = Path('/tmp/mfa_aligned')
LOCAL_TMP     = Path('/tmp/mfa_tmp')

for d in (CSV_DIR, TG_DIR, LOCAL_CORPUS, LOCAL_ALIGNED, LOCAL_TMP):
    d.mkdir(parents=True, exist_ok=True)

with open(WHISPER_CACHE, encoding='utf-8') as f:
    whisper_results = json.load(f)

n_diar = sum(1 for d in whisper_results.values()
             if any(s.get('speaker') for s in d['segments']))
print(f'Archivos en caché : {len(whisper_results)}')
print(f'Con diarización   : {n_diar}')
print(f'WAVs en Drive     : {len(list(SEGMENTS_DIR.glob("*_FULL.wav")))}')

✅ Paquetes Python instalados
Mounted at /content/drive
✅ Imports OK
Archivos en caché : 1
Con diarización   : 1
WAVs en Drive     : 1


In [4]:
# ── CELDA 3: funciones auxiliares ─────────────────────────────────────────────

def sanitize(t):
    return t.replace('"', '""').strip()

def build_ip(word_sn_list, thr):
    SIL = {'', '<eps>', 'sp', 'sil', 'spn'}
    words = [(e.start, e.end, e.label) for e in word_sn_list
             if e.label.strip() not in SIL]
    if not words: return []
    phrases, ps, pe, pw = [], words[0][0], words[0][1], [words[0][2]]
    for i in range(1, len(words)):
        if words[i][0] - words[i-1][1] >= thr:
            phrases.append((ps, pe, ' '.join(pw)))
            ps, pw = words[i][0], [words[i][2]]
        else:
            pw.append(words[i][2])
        pe = words[i][1]
    phrases.append((ps, pe, ' '.join(pw)))
    return phrases

def count_vowel_phones(phone_entries, ps, pe, tol=0.01):
    count = 0
    for s, e, label in phone_entries:
        if s >= ps - tol and e <= pe + tol:
            base = re.sub(r'[0-9_]+$', '', label.strip())
            if base and VOWEL_RE.match(base):
                count += 1
    return count

def pitch_range_fn(snd, s, e):
    """s y e en el timeline del _FULL.wav (ya ajustados, sin time_offset)."""
    part = snd.extract_part(from_time=s, to_time=e, preserve_times=False)
    pw   = part.to_pitch_ac(0.01, pitch_floor=PITCH_FLOOR, pitch_ceiling=PITCH_CEILING)
    f0w  = pw.selected_array['frequency']
    vw   = f0w[f0w > 0]
    if len(vw) < MIN_VOICED:
        return None, None, None, len(vw)
    q1, q3   = float(np.percentile(vw, 25)), float(np.percentile(vw, 75))
    floor_ad = max(50.0, q1 * 0.75)
    ceil_ad  = min(800.0, q3 * 1.50)
    pitch = part.to_pitch_ac(0.01, pitch_floor=floor_ad, pitch_ceiling=ceil_ad)
    f0    = pitch.selected_array['frequency']
    vcd   = f0[f0 > 0]
    if len(vcd) < MIN_VOICED:
        return None, None, None, len(vcd)
    st = 12.0 * np.log2(vcd)
    lo, hi = float(np.percentile(st, P_LOW)), float(np.percentile(st, P_HIGH))
    return round(hi - lo, 3), round(lo, 3), round(hi, 3), len(vcd)

def _get_pauses(ips):
    return [round(ips[j]['start'] - ips[j-1]['end'], 4)
            for j in range(1, len(ips)) if ips[j]['start'] > ips[j-1]['end']]

def _make_turn_row(stem, t_id, ips):
    spk    = ips[0]['speaker']
    ts, te = ips[0]['start'], ips[-1]['end']
    tdur   = round(te - ts, 4)
    nvow   = sum(ip['n_vowels'] for ip in ips)
    sptime = round(sum(ip['ip_dur'] for ip in ips), 4)
    pauses = [round(ips[j]['start'] - ips[j-1]['end'], 4)
              for j in range(1, len(ips)) if ips[j]['start'] > ips[j-1]['end']]
    return {
        'file': stem, 'speaker': spk, 'turn_id': t_id,
        'turn_start': ts, 'turn_end': te, 'turn_duration_s': tdur,
        'n_ips': len(ips), 'n_vowels': nvow,
        'speech_time_s': sptime,
        'pause_time_s': round(tdur - sptime, 4),
        'n_pauses': len(pauses),
        'mean_pause_s': round(sum(pauses)/len(pauses), 4) if pauses else 0.0,
        'speech_rate_vps': round(nvow/tdur, 3) if tdur > 0 else None,
        'speaking_proportion': round(sptime/tdur, 3) if tdur > 0 else None,
        'text': ' | '.join(ip['text'] for ip in ips),
    }

def _make_speaker_row(stem, speaker, ip_rows, turn_rows, all_pauses):
    def _r(v): return round(float(v), 3)
    art_rates    = [float(r['speech_rate_vps']) for r in ip_rows
                    if str(r.get('speech_rate_vps','')).strip() not in ('','None')]
    n_vow_per_ip = [int(r['n_vowel_phones']) for r in ip_rows
                    if str(r.get('n_vowel_phones','')).strip() not in ('','None')]
    p_ranges     = [float(r['pitch_range_st']) for r in ip_rows
                    if str(r.get('pitch_range_st','')).strip() not in ('','None','0')]
    p10_vals     = [float(r[f'pitch_P{P_LOW}_st']) for r in ip_rows
                    if str(r.get(f'pitch_P{P_LOW}_st','')).strip() not in ('','None')]
    p90_vals     = [float(r[f'pitch_P{P_HIGH}_st']) for r in ip_rows
                    if str(r.get(f'pitch_P{P_HIGH}_st','')).strip() not in ('','None')]
    pitch_levels = [(lo+hi)/2 for lo, hi in zip(p10_vals, p90_vals)]
    syl_slope = ''
    if len(n_vow_per_ip) >= 3:
        x = np.arange(len(n_vow_per_ip), dtype=float)
        syl_slope = round(float(np.polyfit(x, n_vow_per_ip, 1)[0]), 4)
    n_vowels_total = sum(int(r['n_vowels']) for r in turn_rows)
    speech_time_s  = sum(float(r['speech_time_s']) for r in turn_rows)
    pause_time_s   = sum(float(r['pause_time_s']) for r in turn_rows)
    total_dur_s    = speech_time_s + pause_time_s
    pause_ms       = [p * 1000 for p in all_pauses]
    return {
        'file': stem, 'speaker': speaker,
        'n_ips': len(ip_rows), 'n_turns': len(turn_rows),
        'n_vowels_total': n_vowels_total,
        'art_rate_median': _r(np.median(art_rates))    if art_rates    else '',
        'art_rate_mean':   _r(np.mean(art_rates))      if art_rates    else '',
        'art_rate_sd':     _r(np.std(art_rates))       if art_rates    else '',
        'sylls_per_ip_median': _r(np.median(n_vow_per_ip)) if n_vow_per_ip else '',
        'syl_slope': syl_slope,
        'speech_time_s':    round(speech_time_s, 3),
        'pause_time_s':     round(pause_time_s, 3),
        'total_duration_s': round(total_dur_s, 3),
        'n_pauses':         len(all_pauses),
        'pause_median_ms':  _r(np.median(pause_ms)) if pause_ms else '',
        'pause_mean_ms':    _r(np.mean(pause_ms))   if pause_ms else '',
        'pause_sd_ms':      _r(np.std(pause_ms))    if pause_ms else '',
        'log_pause_median_ms': _r(np.log(np.median(pause_ms))) if pause_ms else '',
        'pauses_per_100_vowels': round(len(all_pauses)/n_vowels_total*100, 2) if n_vowels_total else '',
        'overall_art_rate': _r(n_vowels_total/speech_time_s) if speech_time_s else '',
        'overall_spk_rate': _r(n_vowels_total/total_dur_s)   if total_dur_s   else '',
        'pitch_range_median': _r(np.median(p_ranges))     if p_ranges     else '',
        'pitch_range_mean':   _r(np.mean(p_ranges))       if p_ranges     else '',
        'pitch_range_sd':     _r(np.std(p_ranges))        if p_ranges     else '',
        'pitch_level_median': _r(np.median(pitch_levels)) if pitch_levels else '',
    }

print('✅ Funciones cargadas')

✅ Funciones cargadas


In [5]:
# ── CELDA 4: bucle principal — un fichero a la vez ────────────────────────────
# Reanuda automáticamente si se interrumpe (salta CSVs ya existentes)

stems = sorted(whisper_results.keys())
print(f'Total ficheros: {len(stems)}')
completados = len(list(CSV_DIR.glob('*_spk.csv')))
print(f'Ya completados: {completados}')

for idx, stem in enumerate(stems, 1):
    ip_csv   = CSV_DIR / f'{stem}_ips.csv'
    turn_csv = CSV_DIR / f'{stem}_turns.csv'
    spk_csv  = CSV_DIR / f'{stem}_spk.csv'
    if ip_csv.exists() and turn_csv.exists() and spk_csv.exists():
        print(f'  [{idx:3}/{len(stems)}] {stem}: ya existe → saltando')
        continue

    print(f'\n{"─"*45}')
    print(f'  [{idx:3}/{len(stems)}] {stem}')

    data = whisper_results[stem]
    segs = data['segments']
    to   = data['time_offset']
    fd   = data['full_duration']

    full_wav = SEGMENTS_DIR / f'{stem}_FULL.wav'
    if not full_wav.exists():
        print(f'  ⚠️  {stem}_FULL.wav no encontrado → saltando')
        continue

    y, sr = sf.read(str(full_wav))
    print(f'  Audio: {len(y)/sr/60:.1f} min @ {sr}Hz')

    # ── Segmentar corpus para MFA (en /tmp, rápido) ───────────────────────────
    rec_dir = LOCAL_CORPUS / stem
    if rec_dir.exists(): shutil.rmtree(rec_dir)
    rec_dir.mkdir()

    mapping = []
    for i, seg in enumerate(segs):
        text = seg.get('text', '').strip()
        if not text: continue
        s0_full = max(0.0, seg['start'])
        e0_full = min(len(y)/sr, seg['end'])
        s_samp  = int(s0_full * sr)
        e_samp  = int(e0_full * sr)
        clip    = y[s_samp:e_samp]
        if len(clip) < int(0.1 * sr): continue
        spk_label = (seg.get('speaker') or 'UNKNOWN')
        fname     = f'{stem}_{spk_label}_{i:04d}'
        sf.write(str(rec_dir / f'{fname}.wav'), clip, sr)
        (rec_dir / f'{fname}.lab').write_text(text, encoding='utf-8')
        mapping.append((fname, s0_full, spk_label))
    print(f'  Corpus: {len(mapping)} utterances')

    # ── MFA: alineación para esta grabación ──────────────────────────────────
    stem_aligned = LOCAL_ALIGNED / stem
    stem_aligned.mkdir(exist_ok=True)
    mfa_tmp = LOCAL_TMP / stem
    mfa_tmp.mkdir(exist_ok=True)

    proc = subprocess.run([
        MFA_BIN, 'align',
        str(rec_dir), MFA_DICTIONARY, MFA_ACOUSTIC, str(stem_aligned),
        '--clean', '--include_original_text',
        '--output_format', 'long_textgrid',
        '--temporary_directory', str(mfa_tmp),
        '-j', str(MFA_JOBS),
    ], capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'  ⚠️  MFA: {proc.stderr.strip()[-200:]}')

    # ── Recopilar palabras y fonemas MFA ─────────────────────────────────────
    words_by_spk, phones_by_spk = {}, {}
    n_aligned = 0
    for fname, s0_full, speaker in mapping:
        mfa_path = stem_aligned / f'{fname}.TextGrid'
        if not mfa_path.exists(): continue
        n_aligned += 1
        total_off = s0_full + to   # desplazamiento al timeline del fichero original
        mfa_tg = tg_module.openTextgrid(str(mfa_path), includeEmptyIntervals=True)
        for name in mfa_tg.tierNames:
            nl   = name.lower()
            tier = mfa_tg.getTier(name)
            shifted = [(round(e.start + total_off, 4),
                        round(e.end   + total_off, 4),
                        sanitize(e.label)) for e in tier.entries]
            if 'word'  in nl: words_by_spk.setdefault(speaker, []).extend(shifted)
            elif 'phone' in nl: phones_by_spk.setdefault(speaker, []).extend(shifted)

    for spk in words_by_spk:  words_by_spk[spk].sort(key=lambda x: x[0])
    for spk in phones_by_spk: phones_by_spk[spk].sort(key=lambda x: x[0])
    print(f'  MFA: {n_aligned}/{len(mapping)} alineados')

    # ── Construir TextGrid final ──────────────────────────────────────────────
    segs_shifted = [{'start': round(s['start']+to, 4), 'end': round(s['end']+to, 4),
                     'text': s.get('text',''), 'speaker': s.get('speaker','')} for s in segs]
    whisper_entries = []
    for seg in segs_shifted:
        s = round(max(0.0, seg['start']), 4)
        e = round(min(fd,  seg['end']),   4)
        if e > s: whisper_entries.append((s, e, sanitize(seg.get('text','').strip())))

    tg = tg_module.Textgrid()
    tg.addTier(IntervalTier('transcripcion_whisper',
                            sorted(whisper_entries, key=lambda x: x[0]),
                            minT=0, maxT=fd))
    all_speakers = sorted(set(list(words_by_spk) + list(phones_by_spk)))
    for spk in all_speakers:
        spk_label = spk if spk else 'UNKNOWN'
        if spk in words_by_spk:
            tg.addTier(IntervalTier(f'palabras_mfa_{spk_label}',
                                    words_by_spk[spk], minT=0, maxT=fd))
        if spk in phones_by_spk:
            tg.addTier(IntervalTier(f'fonemas_mfa_{spk_label}',
                                    phones_by_spk[spk], minT=0, maxT=fd))
        if spk in words_by_spk:
            sw = [SimpleNamespace(start=s, end=e, label=l)
                  for s, e, l in words_by_spk[spk]]
            ip = build_ip(sw, PAUSE_THRESHOLD)
            tg.addTier(IntervalTier(f'intonational_phrase_{spk_label}',
                                    ip, minT=0, maxT=fd))
            print(f'  {spk_label}: {len(ip)} IPs')

    try:
        tg.save(str(TG_DIR / f'{stem}.TextGrid'),
                format='long_textgrid', includeBlankSpaces=True)
    except OSError as ex:
        print(f'  ⚠️  TextGrid no guardado: {ex}')

    # ── Análisis prosódico ────────────────────────────────────────────────────
    ip_tiers = [(n.replace('intonational_phrase_',''), tg.getTier(n))
                for n in tg.tierNames if n.startswith('intonational_phrase_')]

    snd  = parselmouth.Sound(str(full_wav))
    n_ok = 0
    stem_rows, stem_turn_rows, stem_ips = [], [], []

    for spk_label, ip_tier in ip_tiers:
        phone_entries = phones_by_spk.get(spk_label, [])
        for i, entry in enumerate(ip_tier.entries):
            ps, pe, pt_ = float(entry.start), float(entry.end), entry.label.strip()
            pd = round(pe - ps, 4)
            if not pt_ or pd < 0.1: continue
            sc = []
            for seg in segs:
                for w in seg.get('words', []):
                    ws = round(w.get('start',-1) + to, 4)
                    we = round(w.get('end',  -1) + to, 4)
                    if ws >= ps - 0.05 and we <= pe + 0.05:
                        sv = w.get('score', w.get('probability'))
                        if sv is not None: sc.append(float(sv))
            conf   = sum(sc)/len(sc) if sc else None
            passes = conf is not None and conf >= CONFIDENCE_THR
            nvow   = count_vowel_phones(phone_entries, ps, pe)
            rate   = round(nvow/pd, 3) if pd > 0 else None
            if passes:
                # Ajustar al timeline del _FULL.wav restando time_offset
                ps_local = max(0.0, ps - to)
                pe_local = min(len(y)/sr, pe - to)
                pr, lo, hi, nv = pitch_range_fn(snd, ps_local, pe_local)
                n_ok += 1
            else:
                pr = lo = hi = nv = None
            stem_rows.append({
                'file': stem, 'speaker': spk_label, 'phrase_id': i+1,
                'start': ps, 'end': pe, 'duration_s': pd, 'text': pt_,
                'n_vowel_phones': nvow, 'speech_rate_vps': rate,
                'confidence': round(conf,4) if conf is not None else '',
                'passes_threshold': passes, 'voiced_frames': nv or '',
                f'pitch_P{P_LOW}_st':  lo or '',
                f'pitch_P{P_HIGH}_st': hi or '',
                'pitch_range_st': pr or '',
            })
            stem_ips.append({'speaker': spk_label, 'start': ps, 'end': pe,
                             'text': pt_, 'n_vowels': nvow, 'ip_dur': pd})

    stem_ips.sort(key=lambda x: x['start'])
    pauses_by_spk = {}
    if stem_ips:
        t_id, current = 1, [stem_ips[0]]
        for ip in stem_ips[1:]:
            if ip['speaker'] == current[-1]['speaker']:
                current.append(ip)
            else:
                pauses_by_spk.setdefault(current[0]['speaker'], []).extend(_get_pauses(current))
                stem_turn_rows.append(_make_turn_row(stem, t_id, current))
                t_id += 1; current = [ip]
        pauses_by_spk.setdefault(current[0]['speaker'], []).extend(_get_pauses(current))
        stem_turn_rows.append(_make_turn_row(stem, t_id, current))

    stem_speaker_rows = []
    if stem_rows:
        for spk in sorted(set(r['speaker'] for r in stem_rows)):
            stem_speaker_rows.append(_make_speaker_row(
                stem, spk,
                [r for r in stem_rows      if r['speaker'] == spk],
                [r for r in stem_turn_rows if r['speaker'] == spk],
                pauses_by_spk.get(spk, [])
            ))

    print(f'  {len(stem_rows)} IPs | {n_ok} con confianza ≥ {CONFIDENCE_THR}')

    # ── Guardar CSVs en Drive ─────────────────────────────────────────────────
    if stem_rows:
        with open(ip_csv, 'w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=list(stem_rows[0].keys()))
            w.writeheader(); w.writerows(stem_rows)
    if stem_turn_rows:
        with open(turn_csv, 'w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=list(stem_turn_rows[0].keys()))
            w.writeheader(); w.writerows(stem_turn_rows)
    if stem_speaker_rows:
        with open(spk_csv, 'w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=list(stem_speaker_rows[0].keys()))
            w.writeheader(); w.writerows(stem_speaker_rows)
    print(f'  → CSV guardado')

    # ── Liberar memoria y limpiar /tmp ────────────────────────────────────────
    del snd, tg, stem_rows, stem_turn_rows, stem_speaker_rows, stem_ips
    gc.collect()
    shutil.rmtree(rec_dir, ignore_errors=True)
    shutil.rmtree(stem_aligned, ignore_errors=True)
    shutil.rmtree(mfa_tmp, ignore_errors=True)

print(f'\n{"="*50}\n✅ Bucle completado')

Total ficheros: 1
Ya completados: 195

─────────────────────────────────────────────
  [  1/1] S1_01
  Audio: 5.4 min @ 16000Hz
  Corpus: 45 utterances
  MFA: 45/45 alineados
  SPEAKER_00: 74 IPs
  SPEAKER_01: 26 IPs
  99 IPs | 52 con confianza ≥ 0.7
  → CSV guardado

✅ Bucle completado


In [ ]:
# ── CELDA 5: combinar CSVs individuales en TSVs maestros ─────────────────────

all_ip_rows, all_turn_rows, all_spk_rows = [], [], []
for f in sorted(CSV_DIR.glob('*_ips.csv')):
    with open(f, encoding='utf-8') as fh:
        all_ip_rows.extend(list(csv.DictReader(fh)))
for f in sorted(CSV_DIR.glob('*_turns.csv')):
    with open(f, encoding='utf-8') as fh:
        all_turn_rows.extend(list(csv.DictReader(fh)))
for f in sorted(CSV_DIR.glob('*_spk.csv')):
    with open(f, encoding='utf-8') as fh:
        all_spk_rows.extend(list(csv.DictReader(fh)))

for rows, name in [
    (all_ip_rows,   'analisis_prosodico.tsv'),
    (all_turn_rows, 'analisis_turnos.tsv'),
    (all_spk_rows,  'analisis_hablante.tsv'),
]:
    if not rows:
        print(f'⚠️  {name}: sin datos')
        continue
    out = DRIVE_BASE / name
    with open(out, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()), delimiter='\t')
        w.writeheader(); w.writerows(rows)
    print(f'✅ {name}: {len(rows)} filas ({len(list(CSV_DIR.glob("*_spk.csv")))} hablantes)')

print(f'\n✅ COMPLETADO')
print(f'   CSVs individuales → {CSV_DIR}')
print(f'   TSVs combinados   → {DRIVE_BASE}')
print(f'   TextGrids         → {TG_DIR}')